# SIH26158 — KABR DJI_0677: adaptive ordered reconstruction

This notebook downloads **DJI_0677.MP4** (about 24,000 source frames) and its DJI telemetry,
then reconstructs the scene from a much smaller, **time-ordered geometry-aware keyframe set**.
The keyframe budget is not fixed: inexpensive capture screening chooses an initial budget, and
reconstruction increases it only when registration or reprojection quality is insufficient.

### Before you start
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Upload `KABR_0677_Dynamic_Keyframes_Colab_Bundle.zip` to MyDrive and extract it there.
   It creates `MyDrive/KABR_0677_Dynamic_Keyframes/` with this notebook and all project files.
3. Run cells **in order**. Cell 2 restarts the runtime once; continue from Cell 3 afterward.

### Outputs
Every attempted budget is retained under a timestamped folder in
`MyDrive/KABR_0677_Dynamic_Keyframes/outputs/kabr_0677_adaptive/`. `OUTPUT` is set to the best acceptable
attempt and contains PLY, LAS, GLB, OBJ, GeoTIFF, reports, and the adaptive decision summary.

In [ ]:
# CELL 1 — verify GPU and mount Google Drive
import os, subprocess
from google.colab import drive

drive.mount('/content/drive')

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
assert gpu.returncode == 0, 'No GPU detected. Select a T4 GPU runtime before continuing.'
print('GPU:', gpu.stdout.strip())

PROJECT_DIR = '/content/drive/MyDrive/KABR_0677_Dynamic_Keyframes'
assert os.path.isfile(f'{PROJECT_DIR}/requirements.txt'), (
    f'Project not found at {PROJECT_DIR}. '
    'Extract KABR_0677_Dynamic_Keyframes_Colab_Bundle.zip in MyDrive first.'
)
print('Project:', PROJECT_DIR)

In [ ]:
# CELL 2 — install Conda (restarts the runtime once; continue from Cell 3)
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# CELL 3 — install CUDA COLMAP and Python dependencies
import os, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/KABR_0677_Dynamic_Keyframes'

assert shutil.which('mamba'), 'Conda is missing. Run Cell 2 and wait for its restart.'

pin = '/usr/local/conda-meta/pinned'
if os.path.exists(pin):
    os.remove(pin)

subprocess.run(
    ['mamba', 'install', '-y', '-q', '-c', 'conda-forge',
     'colmap', 'libfaiss', 'openimageio'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     f'{PROJECT_DIR}/requirements.txt'],
    check=True,
)

check = subprocess.run(['colmap', '-h'], capture_output=True, text=True, errors='replace')
header = (check.stdout or '') + (check.stderr or '')
print('\n'.join(header.splitlines()[:10]))
assert check.returncode == 0 and 'COLMAP' in header, 'COLMAP does not run'
assert 'CUDA' in header, 'This COLMAP build does not report CUDA support'

# Run unit tests
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=PROJECT_DIR, check=True,
)
print('Installation and tests complete.')

In [ ]:
# CELL 4 — download DJI_0677.MP4 and DJI_0677.SRT from HuggingFace
import os, time, urllib.request

HF_BASE = (
    'https://huggingface.co/datasets/imageomics/KABR-raw-videos'
    '/resolve/main/18_01_2023_session_5'
)

INPUT_DIR = '/content/kabr_0677_input'
os.makedirs(INPUT_DIR, exist_ok=True)

VIDEO = f'{INPUT_DIR}/DJI_0677.MP4'
TELEMETRY = f'{INPUT_DIR}/DJI_0677.SRT'

files = {
    VIDEO:     f'{HF_BASE}/DJI_0677.MP4',
    TELEMETRY: f'{HF_BASE}/DJI_0677.SRT',
}

for local_path, url in files.items():
    if os.path.isfile(local_path):
        size_mb = os.path.getsize(local_path) / 1024 / 1024
        print(f'Already exists: {os.path.basename(local_path)} ({size_mb:.1f} MB)')
        continue
    print(f'Downloading {os.path.basename(local_path)} from HuggingFace...')
    t0 = time.time()
    urllib.request.urlretrieve(url, local_path)
    elapsed = time.time() - t0
    size_mb = os.path.getsize(local_path) / 1024 / 1024
    print(f'  Done: {size_mb:.1f} MB in {elapsed:.0f}s '
          f'({size_mb/elapsed:.1f} MB/s)')

print(f'\nVideo:     {VIDEO} ({os.path.getsize(VIDEO)/1024/1024:.1f} MB)')
print(f'Telemetry: {TELEMETRY} ({os.path.getsize(TELEMETRY)/1024:.0f} KB)')

In [ ]:
# CELL 5 — adaptive reconstruction configuration
import os, time

PROJECT_DIR = '/content/drive/MyDrive/KABR_0677_Dynamic_Keyframes'
INPUT_DIR = '/content/kabr_0677_input'
RUN_NAME = 'kabr_0677'
VIDEO = f'{INPUT_DIR}/DJI_0677.MP4'
TELEMETRY = f'{INPUT_DIR}/DJI_0677.SRT'

# Frame-budget controller. A 13-minute flight normally starts near 195 frames
# (one keyframe per four seconds), then grows geometrically only if needed.
MIN_FRAMES = 160
MAX_FRAMES = 600
TARGET_SECONDS_PER_KEYFRAME = 4.0
BUDGET_GROWTH = 1.35
MAX_RECONSTRUCTION_ATTEMPTS = 3

# Explicit mathematical quality targets.
MIN_REGISTRATION_RATIO = 0.90
MAX_REPROJECTION_ERROR_PX = 1.00
MIN_ADAPTIVE_QUALITY_SCORE = 0.90

MAX_WIDTH = 1920
DSM_RESOLUTION_M = 0.5
PROCESSING_PROFILE = 'verified-fast'  # validated runtime/quality profile
AI_MASK_DYNAMIC_OBJECTS = True
RESET_ATTEMPT_WORKSPACE = True

RUN_SESSION = time.strftime('%Y%m%d_%H%M%S')
WORKSPACE_ROOT = f'/content/sih26158_{RUN_NAME}_{RUN_SESSION}'
OUTPUT_ROOT = (
    f'{PROJECT_DIR}/outputs/{RUN_NAME}_adaptive/{RUN_SESSION}'
)

assert os.path.isfile(VIDEO), f'Video not found: {VIDEO}'
assert os.path.isfile(TELEMETRY), f'Telemetry not found: {TELEMETRY}'
assert MIN_FRAMES <= MAX_FRAMES
assert BUDGET_GROWTH > 1.0
print('Video:', VIDEO)
print('Telemetry:', TELEMETRY)
print('Adaptive output root:', OUTPUT_ROOT)

In [ ]:
# CELL 6 — compatibility patch for the supplied DJI SRT spelling
from pathlib import Path

telemetry_py = Path(PROJECT_DIR) / 'sih_drone_pipeline' / 'telemetry.py'
content = telemetry_py.read_text(encoding='utf-8')
patched = content.replace(
    r'(?:longitude|lon|lng)',
    r'(?:longitude|longtitude|lon|lng)',
)
if patched != content:
    telemetry_py.write_text(patched, encoding='utf-8')
    print('Added longtitude compatibility to telemetry.py')
else:
    print('Telemetry spelling compatibility already present')

## Adaptive quality rule

Frames remain in increasing source-time order. Within each temporal bin, the pipeline maximizes
a geometry score combining sharpness, GPS baseline, optical-flow parallax, and surviving tracked
features. Before reconstruction, the controller increases the budget only when overlap is too low;
blur, exposure, GPS jumps, and near-duplicates are reported because blindly adding frames does not
repair them.

For reconstruction attempt \(k\), let \(r_k\) be the registered/selected image ratio,
\(e_k\) the mean reprojection error in pixels, and \(g_k\) the fraction of frame-dependent capture
gates passed. The comparison score is

\[
Q_k = 0.55\min(r_k/0.90,1) +
      0.25\min(1.00/\max(e_k,10^{-6}),1) + 0.20g_k.
\]

An attempt is accepted only when \(r_k\ge0.90\), \(e_k\le1.00\) px, the essential capture gates
pass, and \(Q_k\ge0.90\). Otherwise the next budget is
\(N_{k+1}=\min(600,\lceil1.35N_k\rceil)\). All attempts are retained, and the highest-quality
attempt is selected if the cap is reached.

In [ ]:
# CELL 7 — inexpensive feature/overlap screening before reconstruction
import cv2, json, math, os, subprocess, sys
from pathlib import Path

env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_DIR + os.pathsep + env.get('PYTHONPATH', '')
os.makedirs(OUTPUT_ROOT, exist_ok=True)

cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f'OpenCV could not open {VIDEO}'
source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
source_fps = float(cap.get(cv2.CAP_PROP_FPS))
cap.release()
assert source_frames > 0 and source_fps > 0, 'Invalid video metadata'
duration_s = source_frames / source_fps

def grow_budget(value):
    return min(MAX_FRAMES, max(value + 1, int(math.ceil(value * BUDGET_GROWTH))))

budget = max(MIN_FRAMES, min(MAX_FRAMES, int(math.ceil(duration_s / TARGET_SECONDS_PER_KEYFRAME))))
assessment_history = []

while True:
    assessment_json = f'{OUTPUT_ROOT}/capture_assessment_{budget}.json'
    assessment_workspace = f'{WORKSPACE_ROOT}/assessment_{budget}'
    command = [
        sys.executable, '-m', 'sih_drone_pipeline', 'assess',
        '--video', VIDEO, '--telemetry', TELEMETRY,
        '--workspace', assessment_workspace, '--output', assessment_json,
        '--target-frames', str(budget), '--max-width', str(MAX_WIDTH),
    ]
    print(f'\nScreening {budget} ordered geometry keyframes...')
    completed = subprocess.run(command, cwd=PROJECT_DIR, env=env)
    # Exit code 2 means the assessment produced a valid report with warnings.
    if completed.returncode not in (0, 2):
        raise RuntimeError(f'Capture assessment failed with exit code {completed.returncode}')
    assessment = json.loads(Path(assessment_json).read_text(encoding='utf-8'))
    gates = assessment.get('gates', {})
    metrics = assessment.get('metrics', {})
    entry = {
        'budget': budget,
        'ready': assessment.get('ready'),
        'tracked_fraction_median': metrics.get('tracked_fraction_median'),
        'parallax_px_median': metrics.get('parallax_px_median_at_160x90'),
        'blurred_fraction': metrics.get('blurred_frame_fraction'),
        'poor_exposure_fraction': metrics.get('poor_exposure_fraction'),
        'low_overlap_pass': gates.get('low_overlap_pair_fraction_at_most_10_percent'),
        'near_duplicate_pass': gates.get('near_duplicate_pair_fraction_at_most_25_percent'),
    }
    assessment_history.append(entry)
    print(json.dumps(entry, indent=2))

    # More frames can repair low overlap. Other capture failures require capture/
    # telemetry correction, not an ever-larger reconstruction.
    needs_more_overlap = gates.get('low_overlap_pair_fraction_at_most_10_percent') is False
    if not needs_more_overlap or budget >= MAX_FRAMES:
        break
    next_budget = grow_budget(budget)
    if next_budget == budget:
        break
    budget = next_budget

SELECTED_FRAME_BUDGET = budget
adaptive_screening = {
    'source_frames': source_frames,
    'source_fps': source_fps,
    'duration_s': duration_s,
    'initial_budget_formula': 'clip(ceil(duration_s / target_seconds_per_keyframe), min_frames, max_frames)',
    'selected_initial_reconstruction_budget': SELECTED_FRAME_BUDGET,
    'history': assessment_history,
}
Path(f'{OUTPUT_ROOT}/adaptive_screening.json').write_text(
    json.dumps(adaptive_screening, indent=2), encoding='utf-8'
)
print(f'\nSelected initial reconstruction budget: {SELECTED_FRAME_BUDGET} '
      f'from {source_frames:,} source frames')

In [ ]:
# CELL 8 — bounded adaptive reconstruction; retain the best attempt
import json, math, os, shutil, subprocess, sys, time
from pathlib import Path

def reconstruction_metrics(report, requested_budget):
    sparse = report.get('sparse_metrics', {})
    extraction = report.get('workflow_steps', {}).get('keyframe_extraction', {})
    selected = int(extraction.get('selected_frames') or extraction.get('frames') or requested_budget)
    registered = int(sparse.get('registered_images') or 0)
    reprojection = sparse.get('mean_reprojection_error_px')
    registration_ratio = registered / max(selected, 1)

    capture_gates = report.get('capture_quality', {}).get('gates', {})
    essential_gate_names = [
        'all_frames_readable',
        'blurred_frame_fraction_at_most_10_percent',
        'poor_exposure_fraction_at_most_10_percent',
        'low_overlap_pair_fraction_at_most_10_percent',
    ]
    gate_values = [capture_gates.get(name) is True for name in essential_gate_names]
    gate_fraction = sum(gate_values) / len(gate_values)
    registration_component = min(registration_ratio / MIN_REGISTRATION_RATIO, 1.0)
    reprojection_component = (
        min(MAX_REPROJECTION_ERROR_PX / max(float(reprojection), 1e-6), 1.0)
        if reprojection is not None else 0.0
    )
    score = 0.55 * registration_component + 0.25 * reprojection_component + 0.20 * gate_fraction
    passed = (
        registration_ratio >= MIN_REGISTRATION_RATIO
        and reprojection is not None
        and float(reprojection) <= MAX_REPROJECTION_ERROR_PX
        and all(gate_values)
        and score >= MIN_ADAPTIVE_QUALITY_SCORE
    )
    return {
        'requested_budget': requested_budget,
        'selected_frames': selected,
        'registered_images': registered,
        'registration_ratio': registration_ratio,
        'mean_reprojection_error_px': reprojection,
        'essential_capture_gate_fraction': gate_fraction,
        'near_duplicate_gate_pass': capture_gates.get('near_duplicate_pair_fraction_at_most_25_percent'),
        'quality_score': score,
        'quality_passed': passed,
    }

# One preflight before any expensive work.
preflight_path = f'{OUTPUT_ROOT}/preflight.json'
preflight = [
    sys.executable, '-m', 'sih_drone_pipeline', 'preflight',
    '--video', VIDEO, '--telemetry', TELEMETRY,
    '--workspace', WORKSPACE_ROOT, '--output', preflight_path,
]
assert subprocess.run(preflight, cwd=PROJECT_DIR, env=env).returncode == 0, (
    f'Preflight failed; inspect {preflight_path}'
)

attempts = []
budget = SELECTED_FRAME_BUDGET
for attempt_number in range(1, MAX_RECONSTRUCTION_ATTEMPTS + 1):
    attempt_workspace = f'{WORKSPACE_ROOT}/reconstruction_{budget}'
    attempt_output = f'{OUTPUT_ROOT}/frames_{budget}'
    if RESET_ATTEMPT_WORKSPACE and os.path.isdir(attempt_workspace):
        shutil.rmtree(attempt_workspace)
    os.makedirs(attempt_output, exist_ok=True)

    command = [
        sys.executable, '-m', 'sih_drone_pipeline', 'run',
        '--video', VIDEO, '--telemetry', TELEMETRY,
        '--workspace', attempt_workspace, '--output', attempt_output,
        '--profile', PROCESSING_PROFILE,
        '--target-frames', str(budget),
        '--max-width', str(MAX_WIDTH),
        '--dsm-resolution', str(DSM_RESOLUTION_M),
        '--keyframe-mode', 'geometry',
        '--keyframe-decode-mode', 'sequential',
        '--mapper', 'pose-prior', '--gps-prior-std-m', '5.0',
    ]
    if AI_MASK_DYNAMIC_OBJECTS:
        command.append('--ai-mask-dynamic')

    print(f'\nAdaptive reconstruction attempt {attempt_number}: {budget} frames')
    started = time.time()
    completed = subprocess.run(command, cwd=PROJECT_DIR, env=env)
    elapsed_min = (time.time() - started) / 60.0
    report_path = Path(attempt_output) / 'run_report.json'
    if completed.returncode != 0 or not report_path.is_file():
        raise RuntimeError(
            f'Reconstruction failed for budget {budget}; inspect {attempt_output}/logs '
            'and run_report.partial.json. Frame-budget growth cannot repair an operational failure.'
        )

    report = json.loads(report_path.read_text(encoding='utf-8'))
    metrics = reconstruction_metrics(report, budget)
    metrics.update({
        'attempt': attempt_number,
        'elapsed_minutes': elapsed_min,
        'output': attempt_output,
    })
    attempts.append(metrics)
    print(json.dumps(metrics, indent=2))
    if metrics['quality_passed']:
        print('Quality target achieved; no larger frame set is needed.')
        break
    if budget >= MAX_FRAMES:
        print('Reached MAX_FRAMES; selecting the best measured attempt.')
        break
    budget = grow_budget(budget)

# Never let an added frame budget silently replace a better reconstruction.
best_attempt = max(
    attempts,
    key=lambda item: (item['quality_passed'], item['quality_score'], -item['requested_budget']),
)
OUTPUT = best_attempt['output']
adaptive_decision = {
    'quality_targets': {
        'minimum_registration_ratio': MIN_REGISTRATION_RATIO,
        'maximum_reprojection_error_px': MAX_REPROJECTION_ERROR_PX,
        'minimum_quality_score': MIN_ADAPTIVE_QUALITY_SCORE,
    },
    'budget_rule': f'N_next = min({MAX_FRAMES}, ceil({BUDGET_GROWTH} * N))',
    'attempts': attempts,
    'selected_attempt': best_attempt,
    'quality_achieved': best_attempt['quality_passed'],
    'note': (
        'Reprojection/registration are camera-network diagnostics. Surveyed checkpoints or '
        'reference geometry are still required for an absolute accuracy claim.'
    ),
}
decision_path = Path(OUTPUT_ROOT) / 'adaptive_reconstruction_decision.json'
decision_path.write_text(json.dumps(adaptive_decision, indent=2), encoding='utf-8')
shutil.copy2(decision_path, Path(OUTPUT) / decision_path.name)
print(f'\nSelected output: {OUTPUT}')
if not best_attempt['quality_passed']:
    print('WARNING: the configured quality target was not reached before the safety cap.')

In [ ]:
# CELL 9 — verify the selected reconstruction and summarize evidence
import glob, json, os, subprocess, sys
from pathlib import Path

verify = [
    sys.executable, '-m', 'sih_drone_pipeline', 'verify', '--output', OUTPUT,
]
verification_run = subprocess.run(verify, cwd=PROJECT_DIR, env=env)

report = json.loads(Path(OUTPUT, 'run_report.json').read_text(encoding='utf-8'))
verification_path = Path(OUTPUT, 'verification_report.json')
verification = (
    json.loads(verification_path.read_text(encoding='utf-8'))
    if verification_path.is_file() else {}
)
decision = json.loads(Path(OUTPUT, 'adaptive_reconstruction_decision.json').read_text(encoding='utf-8'))
gpu_summary_path = Path(OUTPUT, 'gpu_usage.summary.json')
gpu = json.loads(gpu_summary_path.read_text(encoding='utf-8')) if gpu_summary_path.is_file() else {}

summary = {
    'source_video_frames': source_frames,
    'selected_keyframes': decision['selected_attempt']['selected_frames'],
    'registered_images': decision['selected_attempt']['registered_images'],
    'registration_ratio': decision['selected_attempt']['registration_ratio'],
    'mean_reprojection_error_px': decision['selected_attempt']['mean_reprojection_error_px'],
    'adaptive_quality_score': decision['selected_attempt']['quality_score'],
    'adaptive_quality_achieved': decision['quality_achieved'],
    'processing_minutes': round(report.get('wall_clock_seconds', 0) / 60, 2),
    'artifact_checks_pass': verification.get('artifact_checks_pass'),
    'production_ready': verification.get('production_ready'),
    'gpu_overall': gpu,
    'selected_output': OUTPUT,
}
print(json.dumps(summary, indent=2))

print('\nOutput files:')
for path in sorted(glob.glob(f'{OUTPUT}/*')):
    size = os.path.getsize(path) if os.path.isfile(path) else 0
    print(f'  {os.path.basename(path):38s} {size/1024/1024:9.1f} MB')

assert decision['quality_achieved'], (
    'Adaptive safety cap reached without meeting reconstruction targets. '
    'Inspect adaptive_reconstruction_decision.json before changing MAX_FRAMES or capture settings.'
)
assert verification_run.returncode == 0, (
    'Artifact verification reported a failure; inspect verification_report.json.'
)

## Interpreting the result

- `adaptive_screening.json` records how the initial ordered-keyframe budget was chosen.
- `adaptive_reconstruction_decision.json` records every attempted budget, the exact score terms,
  and why the selected attempt won.
- A high registration ratio and low reprojection error demonstrate a stable camera network, but do
  **not** independently prove centimetre/metre surface accuracy.
- GPS alignment is diagnostic because the same telemetry is used as a pose prior. Add surveyed GCPs,
  checkpoint distances, or independent reference geometry to claim absolute accuracy.
- Near-duplicate or GPS-jump warnings are deliberately not “fixed” by adding frames. Review the
  capture report when either warning appears.
- For a slower, higher-detail final model, set `PROCESSING_PROFILE = 'high-detail'`; keep the adaptive
  budget controller unchanged.